# Racing Deep Learning

**One model, two teachers.** The same small CNN (`PhysicarNet`) learns to race in two ways: **supervised learning** — a human teaches by example (drive, and every press is a labeled photo), and **reinforcement learning** — a reward teaches by trial and error in the simulator. Both write the same checkpoint (`models/model.pt` + `models/model.onnx`), so you can alternate freely: collect and train by example, refine with RL, collect more examples, and so on. Inference is identical either way — the ONNX file drives the car.

## [0] Model

### 0.1. The action table & the network

One `ACTIONS` list serves everything: the labeling buttons, the classifier's output classes and the RL action space are all indexes into it. `PhysicarNet` takes a raw 160×120 camera frame (normalization and softmax live inside `forward`) and scores each action.

**The shared checkpoint:** every training section below ends by saving `models/model.pt` (weights — what the phases hand to each other) and `models/model.onnx` (the deployable). SL's `RESUME` and RL's `WARM_START` both read `model.pt`, no matter which kind of training wrote it.

In [ ]:
import os

DIR = "assets/racing-deeplearning"
if not os.path.isdir(DIR):               # kernel cwd is the workspace root
    DIR = "examples/" + DIR

import torch
import torch.nn as nn

# The action table — one entry per class/action, shared by BOTH kinds of
# learning: the labeling buttons, the classifier outputs and the RL action
# space are all indexes into this list. Edit freely — add actions, change
# values (hardware limits: |steering| <= 20 deg, speed <= 3.0 m/s).
ACTIONS = [
    {"speed": 0.5, "steering": 20.0},    # 0: left
    {"speed": 0.5, "steering": 0.0},     # 1: straight
    {"speed": 0.5, "steering": -20.0},   # 2: right
]

CAMERA_W, CAMERA_H = 160, 120   # model input resolution (camera is 480x360)
CAMERA_PAN, CAMERA_TILT = 0.0, -15.0   # camera angle (deg) — data collection,
                                       # training and driving must share it


class PhysicarNet(nn.Module):
    """Small CNN: camera image in -> action scores out.
    Normalization lives inside the network, so a raw image goes in."""

    def __init__(self):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 32, 8, 4), nn.ReLU(),
            nn.Conv2d(32, 64, 4, 2), nn.ReLU(),
            nn.Conv2d(64, 64, 3, 1), nn.ReLU(), nn.Flatten())
        with torch.no_grad():
            n = self.cnn(torch.zeros(1, 3, CAMERA_H, CAMERA_W)).shape[1]
        self.head = nn.Sequential(
            nn.Linear(n, 256), nn.ReLU(),
            nn.Linear(256, len(ACTIONS)))

    def forward(self, camera):
        x = camera / 255.0 * 2.0 - 1.0                    # 0-255 -> -1..1
        return torch.softmax(self.head(self.cnn(x)), dim=1)

### 0.2. Talking to the car

The camera in, the wheels and the gimbal out. The API takes radians; we work in degrees (+ = left). The camera viewpoint (`CAMERA_PAN`, `CAMERA_TILT`) must be identical when collecting, training and driving.

In [ ]:
import math
import time

import cv2
import numpy as np
import requests

BASE_URL = "http://localhost"


def camera(width=None, height=None):
    """Latest camera frame as a BGR image (server-resized when asked)."""
    params = {"width": width, "height": height} if width else {}
    jpg = requests.get(f"{BASE_URL}/camera", params=params, timeout=2).content
    return cv2.imdecode(np.frombuffer(jpg, np.uint8), cv2.IMREAD_COLOR)


def drive(speed, steering):
    """speed in m/s, steering in degrees (+ = left). The API wants radians."""
    requests.post(f"{BASE_URL}/speed", json={"value": float(speed)}, timeout=2)
    requests.post(f"{BASE_URL}/steering",
                  json={"value": math.radians(steering)}, timeout=2)


def look(pan, tilt):
    """Point the camera (degrees)."""
    requests.post(f"{BASE_URL}/camera/pan",
                  json={"value": math.radians(pan)}, timeout=2)
    requests.post(f"{BASE_URL}/camera/tilt",
                  json={"value": math.radians(tilt)}, timeout=2)

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(cv2.cvtColor(camera(), cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

### 0.3. The live views

The web view is plumbing, not the logic — run the cell and forget it (the pages themselves live in the example's folder). It skips itself with a notice when port 5000 is already taken by another app.

In [ ]:
"""Live web views (MYAPP tab, port 5000), one page per phase:
labeling counts + last shot, an accuracy curve (SL), a reward chart (RL),
and live action probabilities while driving."""
import socket
import threading

_server = [None]     # the one live web view — a new serve() replaces it


def stop_view():
    if _server[0]:
        _server[0].shutdown()
        _server[0] = None


def _start(app):
    import logging
    logging.getLogger("werkzeug").setLevel(logging.ERROR)
    from werkzeug.serving import make_server
    stop_view()                      # one view at a time — replace the old one
    try:
        probe = socket.socket()                       # quiet pre-check: werkzeug
        probe.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)   # prints and
        probe.bind(("", 5000))                        # exits on a busy port
        probe.close()
        server = make_server("0.0.0.0", 5000, app, threaded=True)
    except (OSError, SystemExit):
        print("web view: port 5000 is in use by another example — "
              "interrupt (⏹) that cell or restart its kernel; "
              "continuing without the live view")
        return None
    _server[0] = server
    threading.Thread(target=server.serve_forever, daemon=True).start()
    print("web view: open the MYAPP tab to watch")
    return server

_counts = [{}]          # photos on disk per action index
_epochs = []            # SL: validation accuracy per epoch
_episodes = []          # RL: reward/steps per finished episode
_status = [""]
_frame = [b""]          # last saved photo as JPEG
_infer = [{}]           # latest inference result

_ACTION_LIST = [{"index": i, **a} for i, a in enumerate(ACTIONS)]


def set_counts(counts):
    _counts[0] = {str(k): v for k, v in counts.items()}


def shot(index, img_bgr):
    _counts[0][str(index)] = _counts[0].get(str(index), 0) + 1
    _frame[0] = cv2.imencode(".jpg", img_bgr)[1].tobytes()


def add_epoch(accuracy):
    _epochs.append({"epoch": len(_epochs) + 1,
                    "accuracy": round(float(accuracy), 3)})


def add_episode(reward, steps):
    _episodes.append({"episode": len(_episodes) + 1,
                      "reward": round(float(reward), 2), "steps": int(steps)})


def set_status(text):
    _status[0] = str(text)


def update(probs, action):
    _infer[0] = {"probs": [round(float(p), 3) for p in probs],
                 "action": int(action)}


def _app(page_file):
    from flask import Flask, Response, jsonify
    app = Flask(__name__)
    page = open(f"{DIR}/webui/{page_file}").read()   # presentation lives in the folder

    @app.get("/")
    def index():
        return page

    @app.get("/frame")
    def frame():
        return Response(_frame[0], mimetype="image/jpeg",
                        headers={"Cache-Control": "no-store"})

    @app.get("/data")
    def data():
        return jsonify({"counts": _counts[0], "epochs": _epochs,
                        "episodes": _episodes, "status": _status[0],
                        "actions": _ACTION_LIST, **_infer[0]})
    return app


def serve_labeling():
    return _start(_app("labeling.html"))


def serve_sl_dashboard():
    return _start(_app("sl-dashboard.html"))


def serve_rl_dashboard():
    return _start(_app("rl-dashboard.html"))


def serve_monitor():
    return _start(_app("monitor.html"))

## [1] Supervised Learning

Behavioral cloning: *the button you press is the answer (y); the photo at that moment is the question (x).*

### 1.1. Labeling

The buttons below drive the car AND save a labeled photo per press (`data/<action index>/`). Each press keeps the car moving until the ~1 s safety watchdog stops it — press repeatedly to keep going, like tapping keys. Collect a few laps: hundreds of photos per class. The **MYAPP tab** shows counts and the last shot.

In [ ]:
from datetime import datetime
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display

for i in range(len(ACTIONS)):
    Path(DIR, "data", str(i)).mkdir(parents=True, exist_ok=True)
session = datetime.now().strftime("%Y%m%d_%H%M%S")  # unique filenames
look(CAMERA_PAN, CAMERA_TILT)   # the model's fixed viewpoint
set_counts({i: len(list(Path(DIR, "data", str(i)).glob("*.jpg")))
            for i in range(len(ACTIONS))})
serve_labeling()                # MYAPP tab: photo counts + last shot

n = 0
status = widgets.Label("The button you press is the answer (y); the photo at that moment is the question (x).")


def snap(i):
    global n
    img = camera()
    cv2.imwrite(f"{DIR}/data/{i}/{session}_{n:06d}.jpg", img)
    n += 1
    shot(i, img)
    drive(ACTIONS[i]["speed"], ACTIONS[i]["steering"])
    status.value = f"[{i}] steering {ACTIONS[i]['steering']:+.0f} — {n:,} photos this session"


buttons = []
for i, a in enumerate(ACTIONS):
    name = "left" if a["steering"] > 0 else "right" if a["steering"] < 0 else "straight"
    b = widgets.Button(description=f"[{i}] {name}", button_style="primary")
    b.on_click(lambda _, k=i: snap(k))
    buttons.append(b)
stop = widgets.Button(description="stop", button_style="danger")
stop.on_click(lambda _: drive(0, 0))

display(widgets.HBox(buttons + [stop]), status)

In [ ]:
from pathlib import Path

for i, a in enumerate(ACTIONS):
    n_disk = len(list(Path(DIR, "data", str(i)).glob("*.jpg")))
    print(f"data/{i}  ({a['steering']:+.0f} deg): {n_disk:,} photos")

### 1.2. Training

20 epochs over your photos (minutes on this CPU). Training photos get random brightness/contrast (the label stays the same) so the model survives lighting changes; validation sees them as-is. Watch the accuracy curve in the **MYAPP tab**.

`RESUME = True` continues from `models/model.pt` — including one that **RL training wrote**: that is the alternation.

In [ ]:
from pathlib import Path

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset

EPOCHS = 20
BATCH_SIZE = 64
LEARNING_RATE = 0.001
VAL_SPLIT = 0.2
RESUME = False      # True: keep training the weights in models/model.pt
                    # (written by EITHER kind of training)


class DrivingData(Dataset):
    def __init__(self, augment=False):
        self.augment = augment
        self.samples = []                       # (photo path, action index)
        for i in range(len(ACTIONS)):
            for f in sorted(Path(DIR, "data", str(i)).glob("*.jpg")):
                self.samples.append((f, i))
        if not self.samples:
            raise SystemExit("data/ is empty — collect photos first (section 1.1)")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        img = cv2.imread(str(path))
        img = cv2.resize(img, (CAMERA_W, CAMERA_H), interpolation=cv2.INTER_AREA)
        if self.augment:    # random brightness/contrast, label stays the same
            img = cv2.convertScaleAbs(img, alpha=np.random.uniform(0.8, 1.2),
                                      beta=np.random.uniform(-30, 30))
        cam = torch.from_numpy(img.transpose(2, 0, 1).astype(np.float32))
        return cam, label


data = DrivingData(augment=True)      # training: random lighting
plain = DrivingData()                 # validation: photos as-is
n_val = max(1, int(len(data) * VAL_SPLIT))
idx = torch.randperm(len(data)).tolist()
train_set = Subset(data, idx[n_val:])
val_set = Subset(plain, idx[:n_val])
train_dl = DataLoader(train_set, BATCH_SIZE, shuffle=True)
val_dl = DataLoader(val_set, BATCH_SIZE)
print(f"training on {len(train_set)} photos, validating on {len(val_set)}"
      f" / {len(ACTIONS)} actions")
serve_sl_dashboard()     # MYAPP tab: accuracy curve per epoch
set_status(f"training on {len(train_set)} photos...")

net = PhysicarNet()
if RESUME and os.path.exists(f"{DIR}/models/model.pt"):
    print("resuming from models/model.pt")
    net.load_state_dict(torch.load(f"{DIR}/models/model.pt"))
opt = torch.optim.Adam(net.parameters(), lr=LEARNING_RATE)

try:
    for epoch in range(EPOCHS):
        net.train()
        for cam, y in train_dl:
            loss = F.nll_loss(net(cam).clamp_min(1e-8).log(), y)
            opt.zero_grad(); loss.backward(); opt.step()

        net.eval()
        correct = total = 0
        with torch.no_grad():
            for cam, y in val_dl:
                correct += (net(cam).argmax(1) == y).sum().item()
                total += len(y)
        print(f"epoch {epoch + 1:2d}/{EPOCHS}  accuracy {correct / total:.3f}")
        add_epoch(correct / total)
        set_status(f"epoch {epoch + 1}/{EPOCHS} · accuracy {correct / total:.3f}")
except KeyboardInterrupt:
    pass                    # interrupt: still export what was trained so far

net.eval()
os.makedirs(f"{DIR}/models", exist_ok=True)
torch.save(net.state_dict(), f"{DIR}/models/model.pt")   # the shared checkpoint
torch.onnx.export(
    net, (torch.zeros(1, 3, CAMERA_H, CAMERA_W),),
    f"{DIR}/models/model.onnx", input_names=["camera"], output_names=["actions"],
    opset_version=17, dynamo=False)
set_status("done — saved models/model.onnx")
stop_view()   # the live view lives and dies with this cell
print("saved -> models/model.pt, models/model.onnx")

### 1.3. Inference

Live inference at 15 Hz: read the camera at model resolution (server-side resize), run the ONNX model, act on the argmax. Runs until you interrupt the cell (⏹) — the `finally` block stops the car.

In [ ]:
import time

import onnxruntime as ort

sess = ort.InferenceSession(f"{DIR}/models/model.onnx",
                            providers=["CPUExecutionProvider"])
look(CAMERA_PAN, CAMERA_TILT)   # the model's fixed viewpoint
serve_monitor()                 # MYAPP tab: action probabilities, live

try:
    while True:
        t0 = time.time()
        img = camera(CAMERA_W, CAMERA_H)
        x = img.transpose(2, 0, 1)[None].astype(np.float32)

        probs = sess.run(None, {"camera": x})[0][0]
        action = int(probs.argmax())
        drive(ACTIONS[action]["speed"], ACTIONS[action]["steering"])
        update(probs, action)

        time.sleep(max(0.0, 1 / 15 - (time.time() - t0)))
except KeyboardInterrupt:
    pass
finally:
    drive(0, 0)
    stop_view()   # the live view lives and dies with this cell
    print("stopped")

## [2] Reinforcement Learning

No labels: the **reward function is the teacher**. The simulator becomes a Gymnasium environment and PPO trains the very same `PhysicarNet` — starting, if you like, from the weights supervised learning just produced. **Simulator only** (training teleports the car constantly).

### 2.1. The environment

Simulator helpers first (pose, teleport, world switching, overlay), then the environment. The pedagogical heart sits at the top of the class: `reward()` scores each step by closeness to the track centerline; `on_reset()` starts every episode 10% further along the lap (rotating training worlds every 10 episodes); an episode dies offtrack/crashed (`is_terminated`) or after 10 s (`is_truncated`). **The reward function is the assignment — edit it here.**

In [ ]:
def sim_status():
    """Simulator status: current world, running/switching flags."""
    return requests.get(f"{BASE_URL}/sim/api/status", timeout=5).json()


def sim_pose(retries=20):
    """Exact vehicle pose {x, y, yaw(rad)} — retries the brief windows where
    the simulator has no pose yet (right after a teleport or world switch)."""
    for _ in range(retries):
        d = requests.get(f"{BASE_URL}/sim/api/pose", timeout=5).json()
        if "x" in d:
            return d
        time.sleep(0.15)
    raise RuntimeError("simulator pose unavailable")


def teleport(x, y, yaw):
    """Place the vehicle at an exact pose (yaw in radians)."""
    requests.post(f"{BASE_URL}/sim/api/pose",
                  json={"x": float(x), "y": float(y), "yaw": float(yaw)},
                  timeout=5)


def overlay(text, ttl=10):
    """Show a status line on the /sim screen (empty text clears it)."""
    requests.post(f"{BASE_URL}/sim/api/overlay",
                  json={"text": str(text), "ttl": ttl}, timeout=2)


def respawn(world=None, start=0.0):
    """Reset the car onto the center line for a fresh run.

    world: switch to this track first (only when different — a switch
           takes seconds, so rotate worlds every N episodes, not every one)
    start: where to start on the track, as a lap fraction (0.0 = start line)
    """
    if world is not None:
        while True:
            s = sim_status()
            if (s.get("running") and s.get("current") == world
                    and not s.get("switching")):
                break
            if not s.get("switching"):    # ask (again) — another script may
                requests.post(f"{BASE_URL}/sim/api/switch",   # have flipped it
                              json={"world": f"{world}.world"}, timeout=5)
            time.sleep(2)

    wp = requests.get(f"{BASE_URL}/sim/api/route", timeout=5).json()["waypoints"]
    i = int(start % 1.0 * (len(wp) - 1))
    yaw = math.atan2(wp[i + 1][1] - wp[i][1], wp[i + 1][0] - wp[i][0])
    teleport(wp[i][0], wp[i][1], yaw)

In [ ]:
import gymnasium as gym
from gymnasium import spaces
from shapely.geometry import Point, Polygon
from shapely.geometry.polygon import LinearRing

STEP_DT = 1 / 15        # one action per camera frame
MAX_STEPS = 150         # episode length limit (10 s at 15 Hz)
WORLDS = ["physicar_base", "2022_june_open"]   # tracks to train on
WHEELBASE = 0.18        # robot dimensions (from its URDF), for wheel positions
TRACK_OF_CAR = 0.16


class PhysicarEnv(gym.Env):

    def reward(self):
        """Score for the step: 1.0 on the center line, 0.0 at the track
        border, scaled by the track width at the nearest waypoint."""
        x, y = self.state["x"], self.state["y"]
        center = self.state["waypoints_center"]
        d = Point(x, y).distance(LinearRing(center))
        i = int(np.argmin(np.hypot(center[:, 0] - x, center[:, 1] - y)))
        half_width = np.hypot(*(self.state["waypoints_outer"][i]
                                - self.state["waypoints_inner"][i])) / 2
        reward = 1.0 - d / half_width
        return float(reward)

    def on_reset(self):
        """Start the next episode on the center line, moving the start
        point 10% along the track each episode so training covers the
        whole lap. The WORLDS tracks take turns every 10 episodes."""
        respawn(world=WORLDS[(self.episode // 10) % len(WORLDS)],
                start=(self.episode * 0.10) % 1.0)

    # ══════════════════ machinery below ══════════════════

    def __init__(self):
        super().__init__()
        self.observation_space = spaces.Box(
            0, 255, (3, CAMERA_H, CAMERA_W), np.float32)
        self.action_space = spaces.Discrete(len(ACTIONS))
        self.episode = -1               # becomes 0 on the first reset()
        self._track_world = None

    # ── track geometry ────────────────────────────────────────────────────

    def _load_track(self):
        current = sim_status().get("current")
        if current == self._track_world:
            return
        r = requests.get(f"{BASE_URL}/sim/api/route", timeout=5).json()
        center = np.asarray(r["waypoints"], float)
        inner = np.asarray(r["inner"], float)
        outer = np.asarray(r["outer"], float)
        if np.allclose(center[0], center[-1]):    # closed loop: drop dup row
            center, inner, outer = center[:-1], inner[:-1], outer[:-1]
        self._waypoints = center
        self._inner_pts = inner
        self._outer_pts = outer
        self._road = Polygon(r["outer"], [r["inner"]])
        self._bounds = requests.get(f"{BASE_URL}/sim/api/bounds", timeout=5).json()
        self._track_world = current

    def _wheel_points(self, x, y, yaw_rad):
        c, s = math.cos(yaw_rad), math.sin(yaw_rad)
        return [Point(x + dx * c - dy * s, y + dx * s + dy * c)
                for dx, dy in ((WHEELBASE / 2, TRACK_OF_CAR / 2),
                               (WHEELBASE / 2, -TRACK_OF_CAR / 2),
                               (-WHEELBASE / 2, TRACK_OF_CAR / 2),
                               (-WHEELBASE / 2, -TRACK_OF_CAR / 2))]

    def _refresh(self):
        self.obs = camera(CAMERA_W, CAMERA_H).transpose(2, 0, 1).astype(np.float32)

        pose = sim_pose()
        odom = requests.get(f"{BASE_URL}/odom", timeout=2).json()
        objects = requests.get(f"{BASE_URL}/sim/api/objects", timeout=2).json().get("objects", [])
        lights = requests.get(f"{BASE_URL}/sim/api/traffic_lights", timeout=2).json().get("lights", [])

        wheels = self._wheel_points(pose["x"], pose["y"], pose["yaw"])
        on_track = [self._road.contains(w) for w in wheels]

        # crashed = a movable object moved since the episode started — only
        # the car can push one (reference poses are snapshotted in reset())
        self._is_crashed = any(
            o["movable"] and math.hypot(
                o["current"]["x"] - self._obj_ref.get(o["name"], o["current"])["x"],
                o["current"]["y"] - self._obj_ref.get(o["name"], o["current"])["y"]) > 0.03
            for o in objects)
        self._is_offtrack = not any(on_track)

        self.state = {
            "x": pose["x"], "y": pose["y"],
            "heading": math.degrees(pose["yaw"]),
            "linear_velocity": odom["velocity"]["linear"],
            "angular_velocity": odom["velocity"]["angular"],
            "waypoints_center": self._waypoints,
            "waypoints_inner": self._inner_pts,
            "waypoints_outer": self._outer_pts,
            "objects": objects,
            "traffic_lights": lights,
            "bounds": self._bounds,
            "steps": self._steps,
            "episode": self.episode,
        }

    def is_terminated(self):
        """The episode dies here: offtrack (all four wheels out) or crashed."""
        return self._is_offtrack or self._is_crashed

    def is_truncated(self):
        """The episode is cut here: the step limit is reached."""
        return self._steps >= MAX_STEPS

    # ── gymnasium interface ───────────────────────────────────────────────

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        drive(0, 0)
        self.episode += 1
        self._steps = 0
        self.on_reset()
        # A world respawn resets the gimbal — restore the model's viewpoint
        look(CAMERA_PAN, CAMERA_TILT)
        time.sleep(0.3)                 # let physics settle
        self._load_track()
        # crash detection baseline: where the objects are right now
        objects = requests.get(f"{BASE_URL}/sim/api/objects", timeout=5).json().get("objects", [])
        self._obj_ref = {o["name"]: o["current"] for o in objects}
        self._refresh()
        self._t_step = time.time()
        return self.obs, {}

    def step(self, action):
        action = int(action)
        drive(ACTIONS[action]["speed"], ACTIONS[action]["steering"])
        # steady step period: sleep whatever remains of STEP_DT after the
        # time already spent since the last step (refresh, reward, inference)
        time.sleep(max(0.0, STEP_DT - (time.time() - self._t_step)))
        self._t_step = time.time()
        self._steps += 1
        self._refresh()

        reward = float(self.reward())
        terminated = self.is_terminated()
        truncated = self.is_truncated()
        overlay(f"episode {self.episode} · step {self._steps}/{MAX_STEPS}"
                f" · reward {reward:+.2f}")
        return self.obs, reward, terminated, truncated, {"action": action}

    def close(self):
        drive(0, 0)
        overlay("")

### 2.2. Training

PPO collects `N_STEPS` of real-time experience (15 Hz — collection alone is ~33 min for 30k steps, realistically 1h+ with resets and policy updates), pausing to update the policy in between (the car stops and the /sim overlay says so). Watch episode rewards climb in the **MYAPP tab**.

- `WARM_START = True` initializes the network from `models/model.pt` — e.g. the one **supervised learning just wrote**: that is the alternation.
- `RESUME = True` continues the exact PPO state from `models/model.zip` instead.
- Interrupting the cell (⏹) still saves and exports what was trained so far.

In [ ]:
import torch

TOTAL_STEPS = 30000      # total experience to train on
N_STEPS = 1500             # experience collected per policy update
BATCH_SIZE = 64
LEARNING_RATE = 0.0003
GAMMA = 0.99               # discount factor: weight of future rewards
RESUME = False             # True: continue the exact PPO state (models/model.zip)
WARM_START = True          # True: start the network from models/model.pt
                           # (written by EITHER kind of training)


class Extractor:
    """Let PPO train our PhysicarNet directly, so what we deploy is exactly
    what was trained. Runs everything except the final action layer
    (mirrors the normalization inside PhysicarNet.forward)."""
    def __new__(cls, *a, **kw):
        from stable_baselines3.common.torch_layers import BaseFeaturesExtractor

        class _E(BaseFeaturesExtractor):
            def __init__(self, observation_space):
                super().__init__(observation_space, features_dim=256)
                self.net = PhysicarNet()

            def forward(self, obs):
                x = obs / 255.0 * 2.0 - 1.0
                return torch.relu(self.net.head[0](self.net.cnn(x)))
        return _E(*a, **kw)


from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.monitor import Monitor


class Dashboard(BaseCallback):
    """Feed the MYAPP dashboard and the /sim overlay while training."""
    def _on_step(self):
        for info in self.locals["infos"]:
            if "episode" in info:   # Monitor adds this when one ends
                add_episode(info["episode"]["r"], info["episode"]["l"])
        set_status(f"collecting experience... "
                   f"{self.num_timesteps:,}/{TOTAL_STEPS:,} steps")
        return True

    def _on_rollout_end(self):
        drive(0, 0)     # don't let the car run blind during the update
        text = (f"updating the policy... "
                f"{self.num_timesteps:,}/{TOTAL_STEPS:,} steps done")
        set_status(text)
        overlay(text, ttl=300)   # a policy update can take a few minutes


# Monitor records every episode (reward, length) to logs/*.monitor.csv
os.makedirs(f"{DIR}/logs", exist_ok=True)
env = Monitor(PhysicarEnv(), time.strftime(f"{DIR}/logs/%Y%m%d-%H%M%S"))
serve_rl_dashboard()

if RESUME and os.path.exists(f"{DIR}/models/model.zip"):
    print("resuming from models/model.zip")
    model = PPO.load(f"{DIR}/models/model.zip", env=env)
else:
    model = PPO("CnnPolicy", env,
                n_steps=N_STEPS, batch_size=BATCH_SIZE,
                learning_rate=LEARNING_RATE, gamma=GAMMA,
                policy_kwargs={"features_extractor_class": Extractor,
                               "normalize_images": False,
                               "net_arch": []},
                verbose=1)
    if WARM_START and os.path.exists(f"{DIR}/models/model.pt"):
        # One model, two teachers: adopt the shared checkpoint (e.g. the one
        # supervised learning wrote) as PPO's starting network.
        sd = torch.load(f"{DIR}/models/model.pt")
        warm = model.policy.features_extractor.net
        warm.load_state_dict(sd)
        with torch.no_grad():
            model.policy.action_net.weight.copy_(warm.head[2].weight)
            model.policy.action_net.bias.copy_(warm.head[2].bias)
        print("warm-started from models/model.pt")
try:
    model.learn(total_timesteps=TOTAL_STEPS, progress_bar=True,
                callback=Dashboard())
except KeyboardInterrupt:
    pass                    # interrupt: still export what was trained so far
env.close()
os.makedirs(f"{DIR}/models", exist_ok=True)
model.save(f"{DIR}/models/model.zip")  # full PPO state, for exact resuming

# Put the trained weights into a plain PhysicarNet and refresh the shared
# checkpoint — supervised learning can now RESUME from it.
net = PhysicarNet()
trained = model.policy.features_extractor.net
net.load_state_dict(trained.state_dict())
with torch.no_grad():   # PPO's action layer becomes the final layer
    net.head[2].weight.copy_(model.policy.action_net.weight)
    net.head[2].bias.copy_(model.policy.action_net.bias)
net.eval()
torch.save(net.state_dict(), f"{DIR}/models/model.pt")   # the shared checkpoint
torch.onnx.export(
    net, (torch.zeros(1, 3, CAMERA_H, CAMERA_W),),
    f"{DIR}/models/model.onnx", input_names=["camera"], output_names=["actions"],
    opset_version=17, dynamo=False)
stop_view()   # the live view lives and dies with this cell
print(f"trained over {env.unwrapped.episode + 1} episodes"
      f" / saved -> models/model.zip, models/model.pt, models/model.onnx")

### 2.3. Inference

Exactly the same cell as 1.3 — same model file, same loop. Whichever teacher trained last is the one driving. Interrupt (⏹) to stop.

In [ ]:
import time

import onnxruntime as ort

sess = ort.InferenceSession(f"{DIR}/models/model.onnx",
                            providers=["CPUExecutionProvider"])
look(CAMERA_PAN, CAMERA_TILT)   # the model's fixed viewpoint
serve_monitor()                 # MYAPP tab: action probabilities, live

try:
    while True:
        t0 = time.time()
        img = camera(CAMERA_W, CAMERA_H)
        x = img.transpose(2, 0, 1)[None].astype(np.float32)

        probs = sess.run(None, {"camera": x})[0][0]
        action = int(probs.argmax())
        drive(ACTIONS[action]["speed"], ACTIONS[action]["steering"])
        update(probs, action)

        time.sleep(max(0.0, 1 / 15 - (time.time() - t0)))
except KeyboardInterrupt:
    pass
finally:
    drive(0, 0)
    stop_view()   # the live view lives and dies with this cell
    print("stopped")